# Step 2 · Documents and RAG
Turn a contract PDF into searchable clauses, then answer questions **with page citations**.

> **Start Jupyter through the project script** so it gets the same settings as the app:
>
> ```bash
> cd copilot
> ./scripts/run.sh jupyter lab
> ```

In [ ]:
import sys, os
sys.path.insert(0, "..")          # make the copilot's `app` package importable
os.chdir("..")                    # run from copilot/, like the app does
print("Models are called at:", os.getenv("BIFROST_URL"), "| profile:", os.getenv("SETTINGS_PROFILE") or "direct to Ollama")

## 1. Upload: PDF → pages → clause chunks → vectors

In [ ]:
from pathlib import Path
from app.ingest.ingest_service import ingest_pdf
pdf = Path("sample_contracts/01_carewell_hospitals.pdf")
doc = ingest_pdf(pdf.name, pdf.read_bytes())
doc

Each chunk keeps its **page** and **clause heading**, so every answer can point to where it came from.

In [ ]:
from app.rag import retriever
question = "What is our liability if there is a data breach?"
for c in retriever.for_question(doc["doc_id"], question):
    print(f"p.{c['page']:<3} {c['heading'][:50]:<52} {c['text'][:70]}…")

## 2. Ask: the model answers only from those clauses

In [ ]:
from app.rag.answerer import answer
print(answer(doc["doc_id"], question)["answer"])

## 3. Honest "Not found"
Ask about something the contract does not cover:

In [ ]:
print(answer(doc["doc_id"], "What is the name of the client's CEO's dog?")["answer"])

## 4. A plain-English summary of the key terms

In [ ]:
from app.rag.summariser import summarise
summary = summarise(doc["doc_id"])
print(summary["overview"])
for s in summary["sections"][:4]:
    print(f"\n{s['name']}: {s['summary']}")

**Takeaway:** retrieval finds the right clauses; the model only reads what it is given, and cites the page.